# Driver Drowsiness Detection - ANN Model
Complete corrected GitHub/Jupyter Notebook JSON file.


In [ ]:
# STEP 1 - IMPORT LIBRARIES
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE = 64
print('All Libraries Imported Successfully')
print('TensorFlow Version:', tf.__version__)


In [ ]:
# STEP 2 - DEFINE DATASET PATHS AND LABELS
source_folders = {
    'eyes/train/Close': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\eyes\\train\\Close',
    'eyes/train/Open': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\eyes\\train\\Open',
    'eyes/val/Close': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\eyes\\val\\Close',
    'eyes/val/Open': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\eyes\\val\\Open',
    'eyes/test/Close': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\eyes\\test\\Close',
    'eyes/test/Open': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\eyes\\test\\Open',
    'yawn/yawn': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\yawn\\yawn',
    'yawn/no_yawn': r'C:\\Users\\semwa\\Project_Bnat\\archive (1)\\data\\yawn\\no yawn'
}

label_map = {
    'eyes/train/Close': 1,
    'eyes/train/Open': 0,
    'eyes/val/Close': 1,
    'eyes/val/Open': 0,
    'eyes/test/Close': 1,
    'eyes/test/Open': 0,
    'yawn/yawn': 1,
    'yawn/no_yawn': 0
}

print('Paths and Labels Defined')
for key, path in source_folders.items():
    print(f'{key} --> EXISTS: {os.path.isdir(path)}')


In [ ]:
# STEP 3 - LOAD IMAGES SMARTLY
X = []
y = []
MAX_PER_FOLDER = 4000

for key, path in source_folders.items():
    if not os.path.isdir(path):
        continue
    label = label_map[key]
    count = 0
    files = [f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))]

    for file in files:
        if count >= MAX_PER_FOLDER:
            break
        file_path = os.path.join(path, file)
        img = cv2.imread(file_path)
        if img is None:
            continue
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = clahe.apply(img)
        X.append(img)
        y.append(label)
        count += 1

X = np.array(X, dtype=np.float32)
y = np.array(y)
print('Normal Images Loaded:', X.shape[0])


In [ ]:
# STEP 4 - NORMALIZE AND FLATTEN
X = X / 255.0
X = X.reshape(X.shape[0], -1)
print('After Flatten:', X.shape)


In [ ]:
# STEP 5 - TRAIN / VAL / TEST SPLIT
X, y = shuffle(X, y, random_state=42)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print('Train:', X_train.shape[0])
print('Val:', X_val.shape[0])
print('Test:', X_test.shape[0])


In [ ]:
# STEP 6 - IMPROVED ANN MODEL
model = Sequential([
    Dense(1024, activation='relu', input_shape=(IMG_SIZE * IMG_SIZE,)),
    Dropout(0.4),
    Dense(512, activation='relu'),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# STEP 7 - TRAIN MODEL
early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
print('Model Training Completed')


In [ ]:
# STEP 8 - PLOT TRAINING ACCURACY AND LOSS
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.legend(['Train','Val'])

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.legend(['Train','Val'])
plt.tight_layout()
plt.show()


In [ ]:
# STEP 9 - EVALUATE ON TEST DATA
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc * 100:.2f}%')
print(f'Test Loss : {test_loss:.4f}')


In [ ]:
# STEP 10 - CLASSIFICATION REPORT AND CONFUSION MATRIX
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred, target_names=['Alert (0)','Drowsy (1)']))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Alert','Drowsy'], yticklabels=['Alert','Drowsy'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


In [ ]:
# STEP 11 - SAVE MODEL
save_path = r'C:\\Users\\semwa\\Project_Bnat\\drowsiness_ann_model.keras'
model.save(save_path)
print(f'Model Saved Successfully at: {save_path}')


In [ ]:
# STEP 12 - PREDICT SINGLE IMAGE
img_path = r'C:\\Users\\semwa\\Project_Bnat\\test.jpg'
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
img = img / 255.0
img = img.reshape(1, -1)
prediction = model.predict(img)[0][0]
label = 'Drowsy' if prediction > 0.5 else 'Alert'
print('Prediction:', label)
print('Confidence:', float(prediction))


In [ ]:
# STEP 13 - REAL TIME WEBCAM TEST
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    face = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
    face = face / 255.0
    face = face.reshape(1, -1)
    pred = model.predict(face, verbose=0)[0][0]
    label = 'Drowsy' if pred > 0.5 else 'Alert'
    cv2.putText(frame, label, (30,40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow('Driver Drowsiness Detection', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
